# Project K, equal battery: MiniCPM-o 2.6 runs the completed Series One battery

**Why this notebook exists.** Kobby's equal-tests ruling (Nov 2,
2026): every model goes through the same Series One battery before
any post speaks across models. This model still owes the blackout, a valid bed, and the muted pairs. This notebook runs the
whole battery for MiniCPM-o 2.6, identical to the other three
equal-battery notebooks: same files, same questions byte for byte,
same repeats, same 33-column results schema as the Series Two runs.

**The battery (41 asks).**

- **Part A (20 asks).** The repaired JSON question on the three
  real clips (vacuum, jar, eggs) and all seven swap files, two
  repeats each.
- **Part A2 (12 asks).** The same question on the three real
  clips in two built versions: muted (picture, no sound) and
  blackout (sound, no picture), two repeats each.
- **Part B (6 asks).** The plain sounds question on the three
  matched bed files (the sentence alone, the sentence over beeps,
  the sentence over the vacuum bed), two repeats each.
- **Part B door (3 asks).** A transcription question on the same
  three matched files, one repeat each. The bed is judged per
  prompt: a model's bed rows are read only through prompts whose
  baseline (the sentence alone) returned the sentence. The door
  gives every model that baseline check in its strongest form.

**What is reused, unchanged.** Step 1 and Step 3 are this model's
Series Two ones, untouched: the sealed private environment from the Series One notebook: `openbmb/MiniCPM-o-2_6` at its pinned hub revision, loaded on the ladder bf16, then 8-bit, then 4-bit, through the model's own unified chat. The runner's ask path,
feed handling, and feed proof are the Series Two ones. Only Step
2's file list and the battery are new.

**The feed proof rule.** Before every answer is taken, the runner
prints the proof for that exact feed, measured on the model's own
feed path. If a track that should carry sound decodes missing,
empty, short, or all zeros, the run hard-stops with a loud banner
and the rows so far are saved. Declared silence (the swap silence
file and the muted versions) is exempt by name in the job list and
is itself verified to be zeros. Nothing is padded quietly.

**How to run it.** Run the cells from the top, in order. Step 1
builds the environment and proves the stack. Step 2 copies the
thirteen files, builds the six muted and blackout versions,
measures them, and saves the built files into the Drive clips
folder. Step 3 writes the runner and runs its check mode (proof,
loading ladder, sanity). Step 4 runs the battery and saves
`equal_minicpm_results.csv` in the Drive clips folder.

## Step 1 · Setup: a private environment for the model's stack

The next cell does four things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to Colab's
   own Python, and it is a tool, not part of the model's stack.
2. Creates a private virtual environment at `/content/minicpm-venv`,
   on Python 3.12, the same Python generation Colab itself runs. The
   model's stack was proven to install cleanly on Python 3.12, so the
   private environment keeps 3.12 and changes only who owns it.
3. Installs the model's stack inside that environment, in two passes.
   First, torch and torchaudio pinned to 2.8.0 from PyTorch's CUDA 12.6
   shelf: that is the exact build this runtime already runs, so the
   chip driver match is the proven one, and the installer cannot drag
   in a multi-gigabyte CUDA 13 torch the driver may not support (left
   floating, the resolver does exactly that). Second, the model card's
   own pin list, unchanged: transformers 4.44.2, accelerate 0.34.2,
   bitsandbytes 0.46.1, librosa 0.10.2.post1 (a newer patch than the
   card lists, because the card's version predates Colab's current
   numpy), soundfile, vocos, and vector-quantize-pytorch. Pillow rides
   along because the model's code and the battery both need it and
   nothing in the pin list pulls it in. vocos and
   vector-quantize-pytorch are parts the model's code imports for its
   speech side; they are installed even though this test never asks
   the model to speak. One more install rides after those two passes:
   flash-attn 2.8.3, the attention library the model's own modeling
   file imports. Transformers checks that file's imports before it
   loads a single weight, so without flash-attn the model cannot
   even start. Colab's stock environment carries it already, which
   is why the model card never lists it; the private environment
   gets its own copy as the official prebuilt wheel for this exact
   stack (Python 3.12, torch 2.8, CUDA 12, cxx11 ABI), pinned by its
   full download address, never a build from source.
4. Proves the stack, in a fresh process under the environment's own
   Python. The proof prints the Python version, the torch version, the
   transformers version and the file it loads from, and it imports
   `WHISPER_ATTENTION_CLASSES` from the transformers whisper module,
   the exact symbol the model's own code asks for and newer
   transformers versions removed, and it imports flash_attn and
   prints its version. A version number alone is not proof anymore;
   the imports are. If any line of the proof is wrong, Step 1 stops
   on the spot and the run goes no further.

Nothing in this cell installs over, downgrades, or removes anything
Colab shipped. The two failed runs of this notebook both died in that
fight; this design simply refuses to have it.

The install downloads a few gigabytes (torch is the big one) and takes
several minutes. That is normal. Every command is printed before it
runs, and the installers stream their output into the cell, so nothing
about this step happens off the record.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. Two
# earlier runs of this notebook died doing exactly that: the pinned
# transformers never landed where the session reads. The stack now
# lives in a private virtual environment at /content/minicpm-venv, and
# Step 1 ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and the one symbol the model's code
# needs are really there.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/minicpm-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

def run_stream(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.run(cmd)
    print("  exit code:", proc.returncode)
    return proc.returncode

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"])
    if rc != 0 or shutil.which("uv") is None:
        print()
        print("!" * 72)
        print("STOP. uv could not be installed, so the private environment")
        print("cannot be built. Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.12. The card's stack
# was proven to install cleanly on Python 3.12 (a full clean resolution
# of the exact pin list), so the private environment keeps 3.12 and
# changes only who owns the environment: this run, not Colab.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--python", "3.12", VENV])
    if rc != 0:
        print()
        print("!" * 72)
        print("STOP. The private environment could not be created.")
        print("Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, pinned to the runtime's own proven build. Colab's
# pip list on the failed runs showed torch 2.8.0+cu126: that is the
# build this chip and driver already run. Left floating, the resolver
# picks a newest torch built on CUDA 13 and downloads gigabytes of
# driver libraries the runtime may not support. Pinning it is not a
# change to the model card's list; the card leaves torch to the
# machine, and this makes the machine's proven choice explicit.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.8.0", "torchaudio==2.8.0",
                 "--index-url", "https://download.pytorch.org/whl/cu126"])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The torch install inside the private environment failed.")
    print("The installer's own output is above. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: torch install failed in venv")

# 3b. The model card's pin list, unchanged, in one resolver pass. No
# force-reinstall: the environment is empty and private, so there is
# nothing to force against. Pillow is added because the model's code
# and the battery import it and no package in the list pulls it in.
CARD_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.46.1",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "vocos==0.1.0",
    "vector-quantize-pytorch==1.18.5",
    "Pillow",
]
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY] + CARD_PACKAGES)
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The model card's stack did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: card stack install failed in venv")

# 3c. flash-attn: the one package the sealed environment was missing.
# MiniCPM's own modeling file imports flash_attn, and transformers
# checks that file's imports before it loads a single weight. The
# fifth round's full-capture log (minicpm_load_log.txt) shows all
# three rungs dying in about three seconds on exactly that check:
# "This modeling file requires the following packages that were not
# found in your environment: flash_attn". Colab's stock environment
# carries flash-attn 2.8.3 already, which is why the model card never
# lists it; the private environment has to be given its own copy. It
# goes in as the official prebuilt wheel, never a build from source:
# Dao-AILab's flash-attn 2.8.3 for CPython 3.12, torch 2.8, and CUDA
# 12, in the cxx11 ABI TRUE form, which is the ABI torch 2.8 itself
# ships with (PyTorch dropped its pre-cxx11 ABI builds at 2.8). The
# address was fetched and the wheel opened when this round was
# built: it answers HTTP 200, it holds the cp312 CUDA extension, and
# its version reads 2.8.3. Its one extra dependency, einops, is
# small and comes from the normal shelf in the same pass.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.8.3/flash_attn-2.8.3"
                    "%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The flash-attn wheel did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: flash-attn install failed in venv")

# 4. Proof, in a fresh process under the environment's Python. Version
# alone is not accepted as proof anymore: the proof also imports
# WHISPER_ATTENTION_CLASSES, the exact symbol MiniCPM's own code asks
# transformers for, whose absence is what a wrong transformers version
# actually breaks, and flash_attn, the package the model's modeling
# file demands at the door (see 3c above).
PROOF = (
    "import sys, torch, transformers\n"
    "print('venv python:', sys.version.split()[0])\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "from transformers.models.whisper.modeling_whisper import "
    "WHISPER_ATTENTION_CLASSES\n"
    "print('WHISPER_ATTENTION_CLASSES import: OK', "
    "sorted(WHISPER_ATTENTION_CLASSES.keys()))\n"
    "import flash_attn\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.44.2" in proof.stdout
      and "WHISPER_ATTENTION_CLASSES import: OK" in proof.stdout
      and "flash_attn version: 2.8.3" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.44.2, flash_attn")
    print("2.8.3) or one of the proof imports failed: the whisper symbol")
    print("or flash_attn. Do not go on to Step 2. Show this to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.44.2, loaded from the path above,")
print("WHISPER_ATTENTION_CLASSES imports from it, and flash_attn")
print("2.8.3 imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). A T4 (about 16 GB)")
print("cannot hold the model in full form; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Get the clips; build the muted and blackout versions

Everything this run asks about lives in your Google Drive, in the
folder named `project-k-clips`. The next cell links your Drive,
finds that folder, and copies thirteen files to a plain work
folder here:

**The three real clips:**

1. `P02_07_5.mp4`: the vacuum clip.
2. `P32_01_54.mp4`: the jar clip.
3. `own-01.mp4`: your eggs recording.

**The seven swap files** (one still picture, seven soundtracks,
loudness matched at build time): `swap_silence.mp4`,
`swap_speech.mp4`, `swap_beeps.mp4`, `swap_water.mp4`,
`swap_vacuum.mp4`, `swap_sizzle.mp4`, `swap_mixed.mp4`.

**The three matched bed files:**

11. `matched_speech.mp4`: the spoken sentence alone, the baseline.
12. `matched_speech_beeps.mp4`: the sentence over a beep bed.
13. `matched_speech_vacuum.mp4`: the sentence over a vacuum bed.

If any of the thirteen is missing, the cell stops and names it.
Nothing is substituted for a missing file, the matched files
included: a bed test without its exact files is not the bed test.

**Standing rule:** a real clip longer than 20 seconds is cut to
its first 15 seconds (that is the eggs recording), the same
treatment as the earlier notebooks. The swap and matched files
are exact test files and are never cut.

**Then the cell builds six versions** from the real clips as they
stand in the work folder, and measures each one from its decoded
track before printing the measurement:

- **Muted** (`equal_muted_<name>`): the same picture, a silent
  track. The decoded track must measure as silence (RMS at or
  below -90 dB) at the source's duration, or the cell stops.
- **Blackout** (`equal_blackout_<name>`): a plain black picture
  at the clip's own size and length, the original sound
  untouched. The decoded track's duration and RMS must match the
  source clip's (within 0.5 seconds and 1.5 dB), or the cell
  stops.

Built files are saved back into the Drive clips folder, so every
model's run can ride the identical files.

Do not go on until the ready line names all of it: 3 real clips,
7 swap files, 3 matched files, 6 built versions.

In [ ]:
# --- Get the clips; build the muted and blackout versions -----------------------
# Copies thirteen files from the Drive clips folder, applies the
# standing cut rule to the real clips, then builds the six muted
# and blackout versions and measures every built file from its
# decoded track. A missing file stops the cell and names the
# file; a built file that decodes wrong stops the cell too.
import os, shutil, subprocess, json, math
import numpy as np

REAL = ["P02_07_5.mp4", "P32_01_54.mp4", "own-01.mp4"]
SWAP = ["swap_silence.mp4", "swap_speech.mp4", "swap_beeps.mp4",
        "swap_water.mp4", "swap_vacuum.mp4", "swap_sizzle.mp4",
        "swap_mixed.mp4"]
MATCHED = ["matched_speech.mp4", "matched_speech_beeps.mp4",
           "matched_speech_vacuum.mp4"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/project-k-clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/project-k-clips"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

missing = [name for name in REAL + SWAP + MATCHED if not bring(name)]
if missing:
    print("!" * 72)
    print("STOP: these files are missing from the Drive clips folder:")
    for name in missing:
        print("MISSING:", name)
    print("Nothing is substituted for a missing file. Add the exact")
    print("files to the project-k-clips folder, then run this cell again.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2: "
                     + ", ".join(missing))

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

def decode_mono(path, sr=48000):
    proc = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "1",
         "-ar", str(sr), "-f", "f32le", "-"],
        capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(f"decode failed for {path}: "
                           + proc.stderr.decode(errors="replace")[-500:])
    return np.frombuffer(proc.stdout, dtype=np.float32), sr

def rms_db(x):
    if len(x) == 0:
        return -120.0
    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))
    return 20.0 * math.log10(v) if v > 0 else -120.0

def save_both(local_path):
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(local_path, os.path.join(
            DRIVE_CLIPS, os.path.basename(local_path)))
        print("saved to Drive clips folder:",
              os.path.basename(local_path))

def video_size(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        w, h = out.stdout.strip().split(",")
        return int(w), int(h)
    except Exception:
        return None, None

# The standing cut rule, real clips only: longer than 20 seconds
# means cut to the first 15. Swap and matched files are exact
# test files and are never cut.
for name in REAL:
    path = os.path.join(WORK, name)
    d = duration_s(path)
    if d is not None and d > 20:
        short = os.path.join(WORK, "short_" + name)
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path,
                        "-t", "15", "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-c:a", "aac", short],
                       check=False)
        if os.path.exists(short):
            os.replace(short, path)
            print("cut", name, "to its first 15 seconds")

build_log = []

# ---- muted versions -------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_muted_" + name)
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", src,
             "-f", "lavfi", "-i",
             "anullsrc=channel_layout=stereo:sample_rate=48000",
             "-map", "0:v:0", "-map", "1:a:0",
             "-c:v", "copy", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: muted build failed for {name}")
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "muted",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    src_d = duration_s(src)
    if (src_d is None or abs(entry["duration_s"] - src_d) > 0.5
            or entry["audio_rms_db_decoded"] > -90.0):
        raise SystemExit(f"STOP: muted build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("muted:", entry)

# ---- blackout versions ------------------------------------------------------------
for name in REAL:
    src = os.path.join(WORK, name)
    out = os.path.join(WORK, "equal_blackout_" + name)
    w, h = video_size(src)
    d = duration_s(src)
    if w is None or d is None:
        raise SystemExit(f"STOP: cannot read size or duration of {name}")
    if not os.path.exists(out):
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-f", "lavfi",
             "-i", f"color=c=black:s={w}x{h}:r=25:d={d:.3f}",
             "-i", src, "-map", "0:v", "-map", "1:a",
             "-t", f"{d:.3f}", "-c:v", "libx264",
             "-pix_fmt", "yuv420p", "-c:a", "aac", "-ar", "48000",
             "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: blackout build failed for {name}")
    xs, srs = decode_mono(src)
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out), "kind": "blackout",
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "source_rms_db_decoded": round(rms_db(xs), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    if (abs(entry["duration_s"] - d) > 0.5
            or abs(entry["audio_rms_db_decoded"]
                   - entry["source_rms_db_decoded"]) > 1.5):
        raise SystemExit(f"STOP: blackout build for {name} decodes "
                         f"wrong: {entry}")
    build_log.append(entry)
    save_both(out)
    print("blackout:", entry)

if build_log:
    log_path = os.path.join(WORK, "equal_build_log.json")
    with open(log_path, "w") as fh:
        json.dump(build_log, fh, indent=2)
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(log_path, os.path.join(DRIVE_CLIPS,
                                           "equal_build_log.json"))
    print("builds logged:", log_path)

print()
ready = True
for name in REAL + SWAP + MATCHED:
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.1f} MB)")
    else:
        ready = False
        print("MISSING:", name)
for name in REAL:
    for kind in ("muted", "blackout"):
        path = os.path.join(WORK, f"equal_{kind}_" + name)
        if os.path.exists(path):
            print("ready:  ", f"equal_{kind}_" + name,
                  f"({os.path.getsize(path) / 1e6:.1f} MB)")
        else:
            ready = False
            print("MISSING:", f"equal_{kind}_" + name)
if not ready:
    print()
    print("!" * 72)
    print("STOP. Some files are missing. Check the Drive folder name is")
    print("exactly project-k-clips, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2")
print()
print("ready: 3 real clips, 7 swap files, 3 matched files, "
      "6 built versions (3 muted, 3 blackout)")

## Step 3 · Write the runner, then check: proof, ladder, sanity

The model never runs inside this notebook's cells directly. It
runs in a plain script, `/content/minicpm_equal_run.py`, under the private environment Step 1 built, through its own Python (VENV_PY).
The first cell below writes that script. The second runs it in
**check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner imported,
   with the stack pins checked a second time, exactly as the
   Series Two runner proved them.
2. **The loading ladder.** the sealed private environment from the Series One notebook: `openbmb/MiniCPM-o-2_6` at its pinned hub revision, loaded on the ladder bf16, then 8-bit, then 4-bit, through the model's own unified chat. A failed rung prints its
   full traceback, never a cut-short reason, and appends the same
   words to `equal_minicpm_load_log.txt` in the clips folder, so a verbatim record
   survives the session. Whichever rung loads is printed and
   saved with every answer.
3. **The sanity question.** The Series Two gate in its own form:
   one plain text question through the real ask path, and the
   word ready is required in the answer. If the answer comes
   back empty, without the word, or with an error, the script
   stops and Step 4 must not be run. Sound arrival is not this
   gate's job; Step 4's per-ask feed proof carries that.

The ladder, the sanity gate, and the ask path are this model's
Series Two ones, unchanged. Only the battery in Step 4 is new.

In [ ]:
# --- Write the equal battery runner script ----------------------------------------
# The runner holds the ladder, the sanity check, the strict feed
# proof, and the equal battery. It is written to a plain file
# here and only ever run under the private environment's Python.
RUNNER_SOURCE = '\n# minicpm_equal_run.py - the equal battery runner for MiniCPM-o 2.6.\n# Written by project_k_minicpm_equal_colab.ipynb. Run only under the\n# private environment\'s Python (/content/minicpm-venv/bin/python).\n# Modes: "check" runs the proof, the loading ladder, and the sanity\n# question, then stops. "battery" does all of that and then runs the\n# equal battery tests and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nfrom transformers.models.whisper.modeling_whisper import WHISPER_ATTENTION_CLASSES\nprint("WHISPER_ATTENTION_CLASSES import: OK",\n      sorted(WHISPER_ATTENTION_CLASSES.keys()))\nif transformers.__version__ != "4.44.2":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.44.2.")\n    sys.exit(3)\n\n# The yes-score question, settled from the model\'s own code before the\n# run (modeling_minicpmo.py in the pinned revision): MiniCPMO.chat\n# calls generate, _decode requests sequences and hidden states from\n# llm.generate but never output scores, and chat returns only the\n# decoded text. There is no score surface on the chat door, so this\n# constant is False and every CSV row marks the score as not exposed.\nYES_SCORE_EXPOSED = False\nprint("yes token score exposed by model.chat:", YES_SCORE_EXPOSED,\n      "(chat returns decoded text only; output scores are never requested)")\n\n# --- Load the model: the same MiniCPM-o ladder as Series One --------------------\nimport traceback, shutil, datetime\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\n\nREPO = "openbmb/MiniCPM-o-2_6"\nREVISION = "06849bfd36da94da1f5a88fa17e8ba63f08a4c4d"\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/project-k-clips/equal_minicpm_load_log.txt",\n                  "/content/equal_minicpm_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("MiniCPM-o 2.6 Series Two load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\nlog_load("repo: " + REPO + " at pinned revision " + REVISION + "\\n")\n\ntry:\n    from transformers.dynamic_module_utils import HF_MODULES_CACHE\n    module_cache = os.path.join(str(HF_MODULES_CACHE), *REPO.split("/"))\n    if os.path.isdir(module_cache):\n        shutil.rmtree(module_cache)\n        clear_note = "cleared cached remote code for this repo: " + module_cache\n    else:\n        clear_note = ("no cached remote code for this repo to clear: "\n                      + module_cache)\nexcept Exception as e:\n    clear_note = "WARNING: the remote-code cache clear did not run: " + str(e)\nprint(clear_note)\nlog_load(clear_note + "\\n")\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\ntokenizer = None\nMODEL_NAME = None\nLOAD_MODE = None\n\ndef try_load(mode):\n    kwargs = dict(trust_remote_code=True, attn_implementation="sdpa",\n                  init_vision=True, init_audio=True, init_tts=False)\n    if mode == "bf16":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, **kwargs)\n        m = m.eval().cuda()\n    elif mode == "8-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, load_in_8bit=True,\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    elif mode == "4-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16,\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    t = AutoTokenizer.from_pretrained(REPO, revision=REVISION,\n                                      trust_remote_code=True)\n    return m, t\n\nrung_errors = []\nfor mode in ("bf16", "8-bit", "4-bit"):\n    print("=" * 70)\n    print(f"rung: {REPO} in {mode} form")\n    try:\n        m, t = try_load(mode)\n        model, tokenizer = m, t\n        MODEL_NAME, LOAD_MODE = REPO, mode\n        print("loaded:", REPO, "in", mode, "form")\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        rung_errors.append((REPO, mode, tb))\n        print("this rung failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("rung failed: " + REPO + " in " + mode + " form\\n")\n        log_load(tb)\n        model = None\n        tokenizer = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("ALL THREE RUNGS FAILED. MiniCPM-o 2.6 did not load in any fit.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every rung is saved, word for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# --- Sanity check: can the model write at all? ---------------------------------\ndef answer_text(res):\n    if isinstance(res, dict):\n        return str(res.get("text", "") or "")\n    if isinstance(res, (list, tuple)):\n        return answer_text(res[0]) if res else ""\n    return str(res or "")\n\ntry:\n    msgs = [{"role": "user",\n             "content": ["Reply with the word ready."]}]\n    sres = model.chat(msgs=msgs, tokenizer=tokenizer,\n                      sampling=False, max_new_tokens=16)\n    sraw = answer_text(sres).strip()\n    print("SANITY RAW OUTPUT:", repr(sraw))\n    if sraw:\n        print("sanity check passed: the model can write.")\n    else:\n        print("WARNING: the sanity check came back EMPTY.")\n        print("STOP. Show this output to Muse.")\n        sys.exit(5)\nexcept SystemExit:\n    raise\nexcept Exception as e:\n    print("WARNING: the sanity check FAILED with an error:")\n    print(str(e))\n    print("STOP. Show this output to Muse.")\n    sys.exit(5)\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, ladder, and sanity all passed.")\n    print("The equal battery was not run. Run Step 4 for the tests.")\n    sys.exit(0)\n\n# --- Equal battery ---------------------------------------------------------\nimport csv, glob, hashlib, json, math, re, subprocess\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom PIL import Image\n\n# The repaired JSON question, injected at build time from\n# pilot2/run_eggs.py, character for character (444 characters).\nPROMPT_JSON = \'You are given a short first-person video clip. Watch and listen to the whole clip.\\nRespond with ONLY a JSON object in exactly this form, no other text: {"seen": "<the main things happening that you can SEE, short>", "heard": "<the main things happening that you can HEAR, short; write the word none if you hear no sound at all>", "sound_start_s": <number: the second at which the main sound event starts, or the word null if there is no sound>}\'\nassert len(PROMPT_JSON) == 444, f"prompt drifted: {len(PROMPT_JSON)} chars"\n\nCLIPS = "/content/clips"\nTRACK_DIRS = ["/content/drive/MyDrive/project-k-clips/equal_extracted_tracks",\n              "/content/equal_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nFEED_WITH_PICTURE = ("omni chat units: 1 frame per second + the matching 1s "\n                     "of the 16 kHz audio track per unit, frames downscaled "\n                     "to fit 448x448, max 15 units, max_slice_nums=1, "\n                     "greedy (sampling=False), no generated speech")\nFEED_AUDIO_ALONE = ("closest audio-only unified chat form: model.chat "\n                    "accepts numpy audio with no image; the code has no "\n                    "separate audio-only method, so the unified chat is "\n                    "used with the default audio-and-text system prompt, "\n                    "per-second 16 kHz audio chunks, no picture and no "\n                    "unit markers, greedy (sampling=False), "\n                    "no generated speech")\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\nFRAME_DIR = "/content/series2_frames"\nos.makedirs(FRAME_DIR, exist_ok=True)\n_feed_cache = {}\n\ndef extract_feed(path, with_picture, expected_sound, expected_silence):\n    key = (path, bool(with_picture), bool(expected_sound),\n           bool(expected_silence))\n    if key in _feed_cache:\n        return _feed_cache[key]\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    tag = re.sub(r"[^A-Za-z0-9]+", "_",\n                 os.path.basename(path) + "_" + str(with_picture))\n    fdir = os.path.join(FRAME_DIR, tag)\n    os.makedirs(fdir, exist_ok=True)\n    for old in glob.glob(os.path.join(fdir, "*")):\n        os.remove(old)\n\n    frame_rc = None\n    frames = []\n    if with_picture:\n        fproc = subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", path, "-vf", "fps=1",\n             os.path.join(fdir, "f%03d.png")], capture_output=True, text=True)\n        frame_rc = fproc.returncode\n        frames = sorted(glob.glob(os.path.join(fdir, "*.png")))\n        if frame_rc != 0:\n            raise FeedProofError(\n                f"frame extraction failed for {path}: rc={frame_rc}: "\n                + (fproc.stderr or "")[-500:])\n        if not frames:\n            raise FeedProofError(f"frame extraction produced no frames: {path}")\n\n    wav = os.path.join(fdir, "audio.wav")\n    aproc = subprocess.run(\n        ["ffmpeg", "-y", "-v", "error", "-i", path, "-vn", "-ac", "1",\n         "-ar", "16000", wav], capture_output=True, text=True)\n    audio_rc = aproc.returncode\n    if audio_rc != 0:\n        raise FeedProofError(\n            f"audio extraction failed for {path}: rc={audio_rc}: "\n            + (aproc.stderr or "")[-500:])\n    if not os.path.exists(wav):\n        raise FeedProofError(f"audio extraction produced no wav for {path}")\n    audio_np, sr = librosa.load(wav, sr=16000, mono=True)\n    audio_np = np.asarray(audio_np, dtype=np.float32)\n\n    src_dur = media_duration(path)\n    if src_dur is None:\n        raise FeedProofError(f"could not read media duration for {path}")\n    unit_count = int(round(src_dur))\n    if unit_count <= 0:\n        raise FeedProofError(f"media duration rounds to zero units: {path}")\n    if unit_count > 15:\n        unit_count = 15\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"extracted track is empty for {path}")\n    if track_dur < unit_count - 0.5:\n        raise FeedProofError(\n            f"extracted track is short for {path}: {track_dur:.3f}s of "\n            f"track for {unit_count} units")\n\n    if with_picture and len(frames) < unit_count - 1:\n        raise FeedProofError(\n            f"frame extraction is short for {path}: {len(frames)} frames "\n            f"for {unit_count} units")\n\n    per_unit_rms = []\n    per_unit_padding = []\n    zero_chunks = 0\n    for i in range(unit_count):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) == 0:\n            raise FeedProofError(\n                f"unit {i} has no audio samples for {path}")\n        per_unit_rms.append(round(rms_db(chunk), 2))\n        if not np.any(chunk):\n            zero_chunks += 1\n        per_unit_padding.append(int(sr - len(chunk)) if len(chunk) < sr else 0)\n\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    if expected_sound and (nonzero_share == 0.0 or overall_rms <= -119.5\n                           or zero_chunks == unit_count):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for {path}: "\n            f"rms={overall_rms} dB, nonzero share={nonzero_share}")\n    if expected_silence and nonzero_share > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {path}: "\n            f"nonzero share={nonzero_share}")\n\n    track_hash = hashlib.sha256(audio_np.tobytes()).hexdigest()\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                sf.write(tp, audio_np, sr, subtype="PCM_16")\n            track_paths.append(tp)\n        except Exception:\n            pass\n\n    contents = []\n    for i in range(unit_count):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) < sr:\n            chunk = np.pad(chunk, (0, sr - len(chunk))).astype(np.float32)\n        if with_picture:\n            img = Image.open(frames[min(i, len(frames) - 1)]).convert("RGB")\n            img.thumbnail((448, 448))\n            contents.extend(["<unit>", img, chunk])\n        else:\n            contents.append(chunk)\n\n    proof = {\n        "file": os.path.basename(path),\n        "source_file_sha256": file_sha256(path),\n        "frame_ffmpeg_rc": frame_rc,\n        "audio_ffmpeg_rc": audio_rc,\n        "track_duration_s": round(track_dur, 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": overall_rms,\n        "track_nonzero_share": round(nonzero_share, 6),\n        "unit_count": int(unit_count),\n        "per_unit_rms_db": per_unit_rms,\n        "per_unit_padding_samples": per_unit_padding,\n        "padding_samples_total": int(sum(per_unit_padding)),\n        "zero_chunk_count": int(zero_chunks),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n        "feed": FEED_WITH_PICTURE if with_picture else FEED_AUDIO_ALONE,\n    }\n    feed = {"contents": contents, "proof": proof}\n    _feed_cache[key] = feed\n    return feed\n\ndef ask(job, prompt, max_new_tokens):\n    with_picture = job["arm"] != "audio_alone"\n    feed = extract_feed(os.path.join(CLIPS, job["file"]), with_picture,\n                        job["expected_sound"], job["expected_silence"])\n    proof = feed["proof"]\n    print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n    contents = list(feed["contents"]) + [prompt]\n    if job["arm"] == "audio_alone":\n        sys_msg = model.get_sys_prompt(mode="default", language="en")\n        res = model.chat(msgs=[sys_msg, {"role": "user", "content": contents}],\n                         tokenizer=tokenizer, sampling=False,\n                         max_new_tokens=max_new_tokens, omni_input=False,\n                         chunk_input=True, use_tts_template=False,\n                         generate_audio=False, max_slice_nums=1,\n                         use_image_id=False)\n    else:\n        sys_msg = model.get_sys_prompt(mode="omni", language="en")\n        res = model.chat(msgs=[sys_msg, {"role": "user", "content": contents}],\n                         tokenizer=tokenizer, sampling=False,\n                         max_new_tokens=max_new_tokens, omni_input=True,\n                         use_tts_template=False, generate_audio=False,\n                         max_slice_nums=1, use_image_id=False)\n    text = answer_text(res).strip()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text, proof\n\n# --- Test definitions: the equal battery (Series One, completed) -----------------\n# The plain sounds question, P3, injected at build time from\n# pilot2/swap/run_swap.py, character for character (143 characters).\nPROMPT_P3 = "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'."\nassert len(PROMPT_P3) == 143, f"P3 drifted: {len(PROMPT_P3)} chars"\n\n# The transcription door for the bed test. New for this battery,\n# and the same words in all four equal-battery runners: bed\n# validity is judged per prompt, and this door gives every model\n# the baseline check in its strongest form.\nTRANSCRIPT_PROMPT = "Write down any words spoken in this clip. If no words are spoken, say \'no words\'."\n\nSWAP_FILES = {\n    "silence": "swap_silence.mp4",\n    "speech": "swap_speech.mp4",\n    "beeps": "swap_beeps.mp4",\n    "water": "swap_water.mp4",\n    "vacuum": "swap_vacuum.mp4",\n    "sizzle": "swap_sizzle.mp4",\n    "mixed": "swap_mixed.mp4",\n}\n\nREAL_CLIPS = [("vacuum", "P02_07_5.mp4"), ("jar", "P32_01_54.mp4"),\n              ("eggs", "own-01.mp4")]\n\nMATCHED_FILES = [("matched_speech", "matched_speech.mp4"),\n                 ("matched_speech_beeps", "matched_speech_beeps.mp4"),\n                 ("matched_speech_vacuum", "matched_speech_vacuum.mp4")]\n\ndef base_job(test, sound, file, arm, version, expected_sound):\n    return {"test": test, "sound": sound, "file": file, "arm": arm,\n            "version": version,\n            "expected_sound": bool(expected_sound),\n            "expected_silence": not bool(expected_sound)}\n\nJOBS = []\n\n# Part A: the repaired JSON question on the three real clips and\n# all seven swap files, two repeats each (20 asks).\nfor sound, file in REAL_CLIPS:\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\nfor sound in ("silence", "speech", "beeps", "water", "vacuum",\n              "sizzle", "mixed"):\n    for rep in (1, 2):\n        job = base_job("partA_series_one", sound, SWAP_FILES[sound],\n                       "audio_with_video", "original_swap_still",\n                       sound != "silence")\n        JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                     rep, 300))\n\n# Part A2: the same question on the three real clips muted and\n# blacked out, two repeats each (12 asks). The muted job declares\n# silence; the blackout job still carries the sound.\nfor sound, file in REAL_CLIPS:\n    for version, expected in (("muted", False), ("blackout", True)):\n        for rep in (1, 2):\n            job = base_job("partA2_muted_blackout", sound,\n                           f"equal_{version}_{file}",\n                           "audio_with_video", version, expected)\n            JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                         rep, 300))\n\n# Part B: the plain sounds question on the three matched bed\n# files, two repeats each (6 asks). The first file is the\n# baseline, the sentence alone; this prompt\'s bed rows are read\n# only if this prompt\'s baseline returned the sentence.\nfor sound, file in MATCHED_FILES:\n    for rep in (1, 2):\n        job = base_job("partB_matched_bed", sound, file,\n                       "audio_with_video", "original", True)\n        JOBS.append((job, "direct", "plain_sounds", "speech",\n                     PROMPT_P3, rep, 128))\n\n# Part B door: the transcription question on the same three\n# matched files, one repeat each (3 asks). Same reading rule:\n# the door\'s bed rows are read only if the door\'s baseline\n# returned the sentence.\nfor sound, file in MATCHED_FILES:\n    job = base_job("partB_door_transcription", sound, file,\n                   "audio_with_video", "original", True)\n    JOBS.append((job, "direct", "transcription", "speech",\n                 TRANSCRIPT_PROMPT, 1, 128))\n\nprint(f"equal battery jobs: {len(JOBS)} asks "\n      f"(part A: 20, part A2: 12, part B: 6, part B door: 3)")\n\nFIELDS = ["test", "sound", "file", "arm", "version", "question_form",\n          "question_label", "target", "prompt", "repeat",\n          "target_start_s", "expected_sound", "expected_silence",\n          "frame_ffmpeg_rc", "audio_ffmpeg_rc", "track_duration_s",\n          "sample_rate", "sample_count", "track_rms_db",\n          "track_nonzero_share", "unit_count", "per_unit_rms_db",\n          "per_unit_padding_samples", "padding_samples_total",\n          "zero_chunk_count", "track_sha256", "track_copy_paths",\n          "source_file_sha256", "raw_answer", "yes_score",\n          "yes_score_status", "model", "feed"]\n\nrows = []\nfailed = 0\n\ndef save_rows():\n    out_fields = FIELDS\n    for out_path in ("/content/drive/MyDrive/project-k-clips/equal_minicpm_results.csv",\n                     "/content/equal_minicpm_results.csv"):\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=out_fields)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    print("=" * 70)\n    print(f"{job[\'test\']} - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    raw_text = ""\n    error = ""\n    proof = None\n    try:\n        raw, proof = ask(job, prompt, max_tokens)\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    rows.append({\n        "test": job["test"], "sound": job["sound"], "file": job["file"],\n        "arm": job["arm"], "version": job["version"],\n        "question_form": form, "question_label": label, "target": target,\n        "prompt": prompt, "repeat": rep,\n        "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": "not exposed",\n        "yes_score_status": ("model.chat returns decoded text only and "\n                             "does not request output scores"),\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[venv py3.12, torch 2.8.0+cu126]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/minicpm_equal_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/minicpm_equal_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, ladder, sanity -------------------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_equal_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a ladder rung, or the sanity question.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity answered.")
print("Go on to Step 4.")

## Step 4 · Run the equal battery

The next cell runs the runner in **battery mode**: proof and
ladder again, sanity again, then the 41 asks of the equal
battery, the completed Series One battery, identical in all
four equal-battery notebooks. Every ask prints its FEED PROOF
line before the model's answer is taken. Watch for it: that
line is the discipline that makes the answers evidence.

**How this model is asked.** one frame per second with the matching second of sound as unified chat units, frames downscaled to fit 448x448, at most 15 units, no generated speech. Decoding is greedy
everywhere. Token caps per ask are the job list's own: 300 for
the JSON question, 128 for the plain sounds and transcription
questions.

**Part A (20 asks).** The repaired JSON question, word for word
the Series One question, on the three real clips (vacuum, jar,
eggs) and all seven swap files, two repeats each.

**Part A2 (12 asks).** The same question on the three real
clips in the two versions Step 2 built: muted (the picture with
no sound, so any sound the answer reports is invented or
carried by the picture) and blackout (the sound with a black
picture, so the picture cannot help or interfere), two repeats
each.

**Part B (6 asks).** The plain sounds question (the swap
study's P3: "What sounds can you hear in this clip? Describe
only what is audible, without guessing from the pictures. If
the audio is silent, say 'silent'.") on the three matched bed
files, two repeats each.

**Part B door (3 asks).** "Write down any words spoken in this
clip. If no words are spoken, say 'no words'." on the same
three matched files, one repeat each.

**How the bed is read.** Per prompt, never pooled. Each
prompt's bed rows are read only if that same prompt's baseline
(the sentence alone, `matched_speech.mp4`) returned the
sentence. If the plain sounds baseline fails and the
transcription baseline returns the sentence, the bed is read
through the door, and the reverse likewise. Every row stays in
the CSV either way; the rule is applied at write-up time, not
by deleting rows. This replaces the Series One bed design,
where two of three bed tests came out void for want of a
passing baseline: here every model gets the baseline check in
both forms, including its strongest one.

**Hard stops and warnings.** A feed that fails its proof stops
the whole run on the spot (exit code 7), with the rows
completed so far saved first. An ask that errors or comes back
empty is saved with its full error text in `raw_answer` and a
WARNING on screen, never as silence, and the run ends with the
RUN PROBLEM banner (exit code 6). Results save to `equal_minicpm_results.csv` in
the Drive clips folder, one row per ask, with the full
feed-proof numbers on every row, in the same 33-column schema
as the Series Two results. The yes-score columns read n/a on
every row: none of these questions is a yes/no ask.

In [ ]:
# --- Battery mode: the equal battery tests --------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_equal_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

Read this run against this model's own Series One and Series
Two answers, and against the other three models' equal-battery
runs. The battery is identical across the four notebooks: same
files, same questions byte for byte, same repeats. A difference
is a model difference or a feed difference, and the feed
differences are written down, not hidden. This model's feed:
one frame per second with the matching second of sound as unified chat units, frames downscaled to fit 448x448, at most 15 units, no generated speech.

- **Part A against Part A2.** Clean minus muted is what the
  sound adds to the answer. Blackout is what the sound does
  when the picture cannot carry the answer: a model that names
  the sound on the clean clip but not on the blackout was
  reading the picture, not hearing the sound.
- **Part B under the baseline rule.** Read each prompt's bed
  rows only where that prompt's baseline returned the sentence
  (Step 4 states the rule). A bed that bends the transcription
  while the report denies the bed is the bed test's signature
  finding, and it only counts where the baseline passed.
- **The fit.** The `model` column of every row names the rung
  that loaded. A squeezed fit's answers are that fit's answers.
- **Scope.** Every claim stays capped at this model, this
  feed, these files, this date.

When you finish, the file `equal_minicpm_results.csv` in your Drive is the
record. Send it back, and the four equal-battery runs get
written up side by side.